In [1]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 1: Environment and Imports
# ============================================================

import sys
import random
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Python version :", sys.version)
print("PyTorch version:", torch.__version__)
print("NumPy version  :", np.__version__)
print("CUDA available :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device         :", device)

Python version : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
PyTorch version: 2.13.0+cpu
NumPy version  : 2.5.2
CUDA available : False
Device         : cpu


In [2]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 2: Reproducibility Setup
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make PyTorch operations as deterministic as practical
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed:", SEED)
print("Reproducibility setup: PASSED")

Random seed: 42
Reproducibility setup: PASSED


In [3]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 3: Project Root Discovery and Output Directories
# ============================================================

NOTEBOOK_NAME = "03_06_gan_training_experiment_03_semantic_only.ipynb"

# ------------------------------------------------------------
# 1. Search upward from the current working directory
#    for the project root.
# ------------------------------------------------------------

current_path = Path.cwd().resolve()

project_root = None

for candidate in [current_path] + list(current_path.parents):
    if (
        (candidate / "06_cross_modal_learning").exists()
        and (candidate / "06_cross_modal_learning" / "03_cross_modal_learning").exists()
    ):
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'06_cross_modal_learning/03_cross_modal_learning'."
    )

# ------------------------------------------------------------
# 2. Official Phase 3 root
# ------------------------------------------------------------

phase_root = project_root / "06_cross_modal_learning"

official_phase3_root = (
    phase_root / "03_cross_modal_learning"
)

# ------------------------------------------------------------
# 3. Experiment 3 isolated root
# ------------------------------------------------------------

experiment_03_root = (
    official_phase3_root
    / "03_06_gan_training"
    / "experiment_03_semantic_only"
)

checkpoints_dir = experiment_03_root / "checkpoints"
reports_dir = experiment_03_root / "reports"
figures_dir = experiment_03_root / "figures"

# Create directories
for directory in [
    experiment_03_root,
    checkpoints_dir,
    reports_dir,
    figures_dir,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 4. Display paths
# ------------------------------------------------------------

print("Project root:")
print(project_root)

print("\nPhase 3 root:")
print(official_phase3_root)

print("\nExperiment 3 root:")
print(experiment_03_root)

print("\nCheckpoints:")
print(checkpoints_dir)

print("\nReports:")
print(reports_dir)

print("\nFigures:")
print(figures_dir)

# ------------------------------------------------------------
# 5. Safety checks
# ------------------------------------------------------------

assert official_phase3_root.exists()
assert experiment_03_root.exists()
assert checkpoints_dir.exists()
assert reports_dir.exists()
assert figures_dir.exists()

# Verify that Experiment 1's direct output folder is NOT
# being used as Experiment 3's checkpoint location.
experiment_01_checkpoint_dir = (
    official_phase3_root
    / "03_06_gan_training"
    / "checkpoints"
)

assert checkpoints_dir != experiment_01_checkpoint_dir

print("\nOutput directory isolation: PASSED")
print("Experiment 1 checkpoint directory will NOT be modified.")
print("Experiment 2 checkpoint directory will NOT be modified.")

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Phase 3 root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning

Experiment 3 root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_03_semantic_only

Checkpoints:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_03_semantic_only\checkpoints

Reports:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_03_semantic_only\reports

Figures:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_03_semantic_only\figures

Output directory isolation: PASSED
Exper

In [4]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 4: Load Official Phase 02.08 Pair Dataset
# ============================================================

# ------------------------------------------------------------
# Official Phase 02.08 representation-aware pair directory
# ------------------------------------------------------------

pair_dir = (
    phase_root
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)

train_npz_path = (
    pair_dir / "02_08_train_representation_disjoint_pairs.npz"
)

validation_npz_path = (
    pair_dir / "02_08_validation_representation_disjoint_pairs.npz"
)

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------

assert train_npz_path.exists(), (
    f"Training pair file not found:\n{train_npz_path}"
)

assert validation_npz_path.exists(), (
    f"Validation pair file not found:\n{validation_npz_path}"
)

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

train_data = np.load(train_npz_path)

validation_data = np.load(validation_npz_path)

# ------------------------------------------------------------
# Extract arrays
# ------------------------------------------------------------

X_train_source = train_data["source_embeddings"].astype(np.float32)
X_train_target = train_data["target_embeddings"].astype(np.float32)
y_train = train_data["pair_labels"].astype(np.float32)

X_val_source = validation_data["source_embeddings"].astype(np.float32)
X_val_target = validation_data["target_embeddings"].astype(np.float32)
y_val = validation_data["pair_labels"].astype(np.float32)

# ------------------------------------------------------------
# Display shapes
# ------------------------------------------------------------

print("Training source embeddings :", X_train_source.shape)
print("Training target embeddings :", X_train_target.shape)
print("Training labels             :", y_train.shape)

print("\nValidation source embeddings :", X_val_source.shape)
print("Validation target embeddings :", X_val_target.shape)
print("Validation labels             :", y_val.shape)

# ------------------------------------------------------------
# Display dataset paths
# ------------------------------------------------------------

print("\nTraining file:")
print(train_npz_path)

print("\nValidation file:")
print(validation_npz_path)

Training source embeddings : (1776, 32)
Training target embeddings : (1776, 32)
Training labels             : (1776,)

Validation source embeddings : (448, 32)
Validation target embeddings : (448, 32)
Validation labels             : (448,)

Training file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

Validation file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz


In [5]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 5: Dataset Integrity Audit
# ============================================================

EXPECTED_EMBEDDING_DIM = 32
EXPECTED_TRAIN_SIZE = 1776
EXPECTED_VAL_SIZE = 448

# ------------------------------------------------------------
# Basic shape checks
# ------------------------------------------------------------

assert X_train_source.shape == (
    EXPECTED_TRAIN_SIZE,
    EXPECTED_EMBEDDING_DIM
)

assert X_train_target.shape == (
    EXPECTED_TRAIN_SIZE,
    EXPECTED_EMBEDDING_DIM
)

assert y_train.shape == (EXPECTED_TRAIN_SIZE,)

assert X_val_source.shape == (
    EXPECTED_VAL_SIZE,
    EXPECTED_EMBEDDING_DIM
)

assert X_val_target.shape == (
    EXPECTED_VAL_SIZE,
    EXPECTED_EMBEDDING_DIM
)

assert y_val.shape == (EXPECTED_VAL_SIZE,)

# ------------------------------------------------------------
# Check for NaN / Inf
# ------------------------------------------------------------

assert np.isfinite(X_train_source).all()
assert np.isfinite(X_train_target).all()
assert np.isfinite(y_train).all()

assert np.isfinite(X_val_source).all()
assert np.isfinite(X_val_target).all()
assert np.isfinite(y_val).all()

# ------------------------------------------------------------
# Check binary labels
# ------------------------------------------------------------

train_unique_labels = np.unique(y_train)
val_unique_labels = np.unique(y_val)

assert set(train_unique_labels).issubset({0.0, 1.0})
assert set(val_unique_labels).issubset({0.0, 1.0})

# ------------------------------------------------------------
# Class counts
# ------------------------------------------------------------

train_class_0 = int(np.sum(y_train == 0))
train_class_1 = int(np.sum(y_train == 1))

val_class_0 = int(np.sum(y_val == 0))
val_class_1 = int(np.sum(y_val == 1))

# ------------------------------------------------------------
# Numerical summaries
# ------------------------------------------------------------

print("========== DATASET INTEGRITY AUDIT ==========\n")

print("Training samples :", len(y_train))
print("Validation samples:", len(y_val))

print("\nTraining labels:")
print("  Class 0:", train_class_0)
print("  Class 1:", train_class_1)

print("\nValidation labels:")
print("  Class 0:", val_class_0)
print("  Class 1:", val_class_1)

print("\nEmbedding dimension:", EXPECTED_EMBEDDING_DIM)

print("\nTraining source:")
print("  Mean:", float(X_train_source.mean()))
print("  Std :", float(X_train_source.std()))
print("  Min :", float(X_train_source.min()))
print("  Max :", float(X_train_source.max()))

print("\nTraining target:")
print("  Mean:", float(X_train_target.mean()))
print("  Std :", float(X_train_target.std()))
print("  Min :", float(X_train_target.min()))
print("  Max :", float(X_train_target.max()))

print("\nValidation source:")
print("  Mean:", float(X_val_source.mean()))
print("  Std :", float(X_val_source.std()))
print("  Min :", float(X_val_source.min()))
print("  Max :", float(X_val_source.max()))

print("\nValidation target:")
print("  Mean:", float(X_val_target.mean()))
print("  Std :", float(X_val_target.std()))
print("  Min :", float(X_val_target.min()))
print("  Max :", float(X_val_target.max()))

print("\nNaN / Inf check: PASSED")
print("Binary label check: PASSED")
print("Shape check: PASSED")
print("\nDataset integrity audit: PASSED")

========== DATASET INTEGRITY AUDIT ==========

Training samples : 1776
Validation samples: 448

Training labels:
  Class 0: 888
  Class 1: 888

Validation labels:
  Class 0: 224
  Class 1: 224

Embedding dimension: 32

Training source:
  Mean: 0.00288502499461174
  Std : 0.13659241795539856
  Min : -0.7118515968322754
  Max : 1.01650071144104

Training target:
  Mean: 0.49211597442626953
  Std : 2.2497622966766357
  Min : -7.480418682098389
  Max : 10.179366111755371

Validation source:
  Mean: 0.004778953734785318
  Std : 0.14298342168331146
  Min : -0.7478017807006836
  Max : 1.0048245191574097

Validation target:
  Mean: 0.5029195547103882
  Std : 2.223020315170288
  Min : -6.369204044342041
  Max : 8.915579795837402

NaN / Inf check: PASSED
Binary label check: PASSED
Shape check: PASSED

Dataset integrity audit: PASSED


In [6]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 6: Generator Architecture
# ============================================================

class Generator(nn.Module):
    """
    Generator used in Experiments 1, 2, and 3.

    Input:
        32-D source embedding

    Output:
        32-D generated target-like embedding

    Architecture:
        32 → 64 → 128 → 64 → 32
    """

    def __init__(self, input_dim=32, output_dim=32):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.network(x)


# ------------------------------------------------------------
# Instantiate Generator
# ------------------------------------------------------------

generator = Generator(
    input_dim=32,
    output_dim=32
).to(device)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

generator_parameters = sum(
    p.numel()
    for p in generator.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

print("========== GENERATOR ARCHITECTURE ==========\n")

print(generator)

print("\nTotal parameters    :", generator_parameters)
print("Trainable parameters:", trainable_parameters)

# ------------------------------------------------------------
# Forward-pass verification
# ------------------------------------------------------------

test_source = torch.tensor(
    X_train_source[:32],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    test_generated = generator(test_source)

print("\nInput shape :", tuple(test_source.shape))
print("Output shape:", tuple(test_generated.shape))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert generator_parameters == 20768
assert trainable_parameters == 20768
assert test_generated.shape == (32, 32)

print("\nGenerator parameter count: PASSED")
print("Generator forward pass: PASSED")
print("Generator architecture verification: PASSED")

========== GENERATOR ARCHITECTURE ==========

Generator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
  )
)

Total parameters    : 20768
Trainable parameters: 20768

Input shape : (32, 32)
Output shape: (32, 32)

Generator parameter count: PASSED
Generator forward pass: PASSED
Generator architecture verification: PASSED


In [7]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 7: Frozen Baseline Relationship Network
# ============================================================

# ------------------------------------------------------------
# Baseline model
# ------------------------------------------------------------

class BaselineCrossModalModel(nn.Module):
    """
    Official baseline semantic relationship model.

    Input:
        source embedding: 32-D
        target embedding: 32-D

    Interaction representation:
        [S, T, |S-T|, S*T] -> 128-D

    Architecture:
        128 -> 128 -> 64 -> 1
    """

    def __init__(self, embedding_dim=32):
        super().__init__()

        interaction_dim = embedding_dim * 4

        self.classifier = nn.Sequential(
            nn.Linear(interaction_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, source, target):

        absolute_difference = torch.abs(
            source - target
        )

        elementwise_product = (
            source * target
        )

        combined = torch.cat(
            [
                source,
                target,
                absolute_difference,
                elementwise_product
            ],
            dim=1
        )

        logits = self.classifier(combined)

        return logits.squeeze(1)


# ------------------------------------------------------------
# Locate official baseline checkpoint
# ------------------------------------------------------------

baseline_checkpoint_path = (
    official_phase3_root
    / "03_04_baseline_training"
    / "checkpoints"
    / "03_04_baseline_best_model.pt"
)

assert baseline_checkpoint_path.exists(), (
    f"Official baseline checkpoint not found:\n"
    f"{baseline_checkpoint_path}"
)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

baseline_checkpoint = torch.load(
    baseline_checkpoint_path,
    map_location=device,
    weights_only=False
)

print("Baseline checkpoint loaded.")
print("\nCheckpoint keys:")
print(list(baseline_checkpoint.keys()))

# ------------------------------------------------------------
# Recreate exact baseline architecture
# ------------------------------------------------------------

baseline_model = BaselineCrossModalModel(
    embedding_dim=32
).to(device)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"],
    strict=True
)

# ------------------------------------------------------------
# Freeze baseline completely
# ------------------------------------------------------------

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

baseline_model.eval()

# ------------------------------------------------------------
# Parameter counts
# ------------------------------------------------------------

baseline_total_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
)

baseline_trainable_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
    if p.requires_grad
)

print("\n========== FROZEN BASELINE ==========\n")

print(baseline_model)

print("\nTotal parameters    :", baseline_total_parameters)
print("Trainable parameters:", baseline_trainable_parameters)

print("\nOfficial baseline checkpoint:")
print(baseline_checkpoint_path)

print(
    "\nOfficial baseline validation ROC-AUC:",
    baseline_checkpoint["best_val_roc_auc"]
)

# ------------------------------------------------------------
# Forward-pass verification
# ------------------------------------------------------------

test_source = torch.tensor(
    X_train_source[:32],
    dtype=torch.float32,
    device=device
)

test_target = torch.tensor(
    X_train_target[:32],
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    test_logits = baseline_model(
        test_source,
        test_target
    )

print("\nTest source shape :", tuple(test_source.shape))
print("Test target shape :", tuple(test_target.shape))
print("Test logits shape :", tuple(test_logits.shape))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert baseline_total_parameters == 24833
assert baseline_trainable_parameters == 0
assert test_logits.shape == (32,)

print("\nBaseline parameter count: PASSED")
print("Baseline frozen status: PASSED")
print("Baseline forward pass: PASSED")
print("Baseline architecture verification: PASSED")

Baseline checkpoint loaded.

Checkpoint keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_roc_auc', 'seed', 'batch_size', 'learning_rate', 'num_epochs']

========== FROZEN BASELINE ==========

BaselineCrossModalModel(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Total parameters    : 24833
Trainable parameters: 0

Official baseline checkpoint:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training\checkpoints\03_04_baseline_best_model.pt

Official baseline validation ROC-AUC: 0.6626674107142857

Test source shape : (32, 32)
Test target shape : (32, 32)
Test logits shape : (32,)

Baseline parameter count: PASSED
Baseline frozen status: PASSED
Baseline forward pass: PASSED
Baseline 

In [8]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 8: PyTorch Dataset
# ============================================================

class CrossModalGANDataset(Dataset):
    """
    Dataset for cross-modal Generator training.

    Each sample contains:
        source_embedding : 32-D source representation
        target_embedding : 32-D real target representation
        pair_label        : 0 or 1

    The real target is retained because it is part of the
    original pair dataset and will be used for validation
    and diagnostic comparison.

    Generator training itself uses:
        source_embedding
        pair_label
    """

    def __init__(
        self,
        source_embeddings,
        target_embeddings,
        labels
    ):
        self.source_embeddings = torch.tensor(
            source_embeddings,
            dtype=torch.float32
        )

        self.target_embeddings = torch.tensor(
            target_embeddings,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

        assert len(self.source_embeddings) == len(
            self.target_embeddings
        )

        assert len(self.source_embeddings) == len(
            self.labels
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            self.source_embeddings[index],
            self.target_embeddings[index],
            self.labels[index]
        )


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = CrossModalGANDataset(
    X_train_source,
    X_train_target,
    y_train
)

validation_dataset = CrossModalGANDataset(
    X_val_source,
    X_val_target,
    y_val
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("========== DATASET OBJECTS ==========\n")

print("Training dataset size   :", len(train_dataset))
print("Validation dataset size :", len(validation_dataset))

sample_source, sample_target, sample_label = (
    train_dataset[0]
)

print("\nSingle training sample:")
print("  Source shape :", tuple(sample_source.shape))
print("  Target shape :", tuple(sample_target.shape))
print("  Label        :", float(sample_label))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(train_dataset) == 1776
assert len(validation_dataset) == 448

assert sample_source.shape == (32,)
assert sample_target.shape == (32,)
assert sample_label.item() in [0.0, 1.0]

print("\nTraining dataset verification: PASSED")
print("Validation dataset verification: PASSED")
print("Sample structure verification: PASSED")

========== DATASET OBJECTS ==========

Training dataset size   : 1776
Validation dataset size : 448

Single training sample:
  Source shape : (32,)
  Target shape : (32,)
  Label        : 1.0

Training dataset verification: PASSED
Validation dataset verification: PASSED
Sample structure verification: PASSED


In [9]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 9: DataLoaders
# ============================================================

BATCH_SIZE = 32

# ------------------------------------------------------------
# Dedicated generator for deterministic training shuffle
# ------------------------------------------------------------

train_loader_generator = torch.Generator()

train_loader_generator.manual_seed(SEED)

# ------------------------------------------------------------
# Training DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator,
    num_workers=0
)

# ------------------------------------------------------------
# Validation DataLoader
# ------------------------------------------------------------

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ------------------------------------------------------------
# Calculate expected number of batches
# ------------------------------------------------------------

expected_train_batches = int(
    np.ceil(len(train_dataset) / BATCH_SIZE)
)

expected_validation_batches = int(
    np.ceil(len(validation_dataset) / BATCH_SIZE)
)

# ------------------------------------------------------------
# Display information
# ------------------------------------------------------------

print("========== DATALOADERS ==========\n")

print("Batch size:", BATCH_SIZE)

print("\nTraining:")
print("  Samples :", len(train_dataset))
print("  Batches :", len(train_loader))
print("  Expected:", expected_train_batches)
print("  Shuffle :", True)

print("\nValidation:")
print("  Samples :", len(validation_dataset))
print("  Batches :", len(validation_loader))
print("  Expected:", expected_validation_batches)
print("  Shuffle :", False)

# ------------------------------------------------------------
# Test one batch
# ------------------------------------------------------------

batch_source, batch_target, batch_labels = next(
    iter(train_loader)
)

print("\nTraining batch:")
print("  Source :", tuple(batch_source.shape))
print("  Target :", tuple(batch_target.shape))
print("  Labels :", tuple(batch_labels.shape))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(train_loader) == 56
assert len(validation_loader) == 14

assert batch_source.shape == (32, 32)
assert batch_target.shape == (32, 32)
assert batch_labels.shape == (32,)

print("\nTraining DataLoader: PASSED")
print("Validation DataLoader: PASSED")
print("Batch shape verification: PASSED")

========== DATALOADERS ==========

Batch size: 32

Training:
  Samples : 1776
  Batches : 56
  Expected: 56
  Shuffle : True

Validation:
  Samples : 448
  Batches : 14
  Expected: 14
  Shuffle : False

Training batch:
  Source : (32, 32)
  Target : (32, 32)
  Labels : (32,)

Training DataLoader: PASSED
Validation DataLoader: PASSED
Batch shape verification: PASSED


In [10]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 10: Semantic Loss Configuration
# ============================================================

# ------------------------------------------------------------
# Semantic relationship loss
# ------------------------------------------------------------

semantic_criterion = nn.BCEWithLogitsLoss()

# ------------------------------------------------------------
# Experiment 3 objective
# ------------------------------------------------------------

SEMANTIC_LOSS_WEIGHT = 1.0

print("========== SEMANTIC-ONLY LOSS ==========\n")

print("Loss function:")
print("  BCEWithLogitsLoss")

print("\nGenerator objective:")
print("  L_G = 1.0 × L_semantic")

print("\nAdversarial loss:")
print("  DISABLED")

print("\nSemantic loss weight:", SEMANTIC_LOSS_WEIGHT)

# ------------------------------------------------------------
# Numerical loss sanity check
# ------------------------------------------------------------

test_logits = torch.tensor(
    [0.0, 2.0, -2.0, 1.0],
    dtype=torch.float32,
    device=device
)

test_labels = torch.tensor(
    [0.0, 1.0, 0.0, 1.0],
    dtype=torch.float32,
    device=device
)

test_loss = semantic_criterion(
    test_logits,
    test_labels
)

print("\nNumerical sanity check:")
print("  Test loss:", float(test_loss))

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert isinstance(
    semantic_criterion,
    nn.BCEWithLogitsLoss
)

assert SEMANTIC_LOSS_WEIGHT == 1.0
assert torch.isfinite(test_loss)

print("\nLoss function verification: PASSED")
print("Semantic-only objective verification: PASSED")
print("Numerical loss check: PASSED")

========== SEMANTIC-ONLY LOSS ==========

Loss function:
  BCEWithLogitsLoss

Generator objective:
  L_G = 1.0 × L_semantic

Adversarial loss:
  DISABLED

Semantic loss weight: 1.0

Numerical sanity check:
  Test loss: 0.31506624817848206

Loss function verification: PASSED
Semantic-only objective verification: PASSED
Numerical loss check: PASSED


In [11]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 11: Generator Optimizer
# ============================================================

LEARNING_RATE = 0.001

# ------------------------------------------------------------
# Create optimizer for Generator only
# ------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE
)

print("========== GENERATOR OPTIMIZER ==========\n")

print("Optimizer       : Adam")
print("Learning rate   :", LEARNING_RATE)
print("Trainable model : Generator")
print("Loss objective  : Semantic loss only")

# ------------------------------------------------------------
# Verify optimizer
# ------------------------------------------------------------

assert LEARNING_RATE == 0.001

assert len(
    generator_optimizer.param_groups
) == 1

assert (
    generator_optimizer.param_groups[0]["lr"]
    == LEARNING_RATE
)

# Verify all Generator parameters are included
optimizer_parameter_count = sum(
    p.numel()
    for group in generator_optimizer.param_groups
    for p in group["params"]
)

assert optimizer_parameter_count == 20768

print("\nOptimizer parameter count:",
      optimizer_parameter_count)

print("\nOptimizer configuration: PASSED")
print("Generator-only optimization: PASSED")

========== GENERATOR OPTIMIZER ==========

Optimizer       : Adam
Learning rate   : 0.001
Trainable model : Generator
Loss objective  : Semantic loss only

Optimizer parameter count: 20768

Optimizer configuration: PASSED
Generator-only optimization: PASSED


In [12]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 12: Single-Batch Gradient Sanity Check
# ============================================================

# ------------------------------------------------------------
# Reset Generator to a clean state
# ------------------------------------------------------------

generator = Generator(
    input_dim=32,
    output_dim=32
).to(device)

generator.train()

# Recreate optimizer after resetting Generator
generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE
)

# Baseline must remain frozen
baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# ------------------------------------------------------------
# Get one deterministic training batch
# ------------------------------------------------------------

sanity_batch = next(iter(train_loader))

source_batch = sanity_batch[0].to(device)
target_batch = sanity_batch[1].to(device)
label_batch = sanity_batch[2].to(device)

# ------------------------------------------------------------
# Record baseline parameters BEFORE update
# ------------------------------------------------------------

baseline_before = {
    name: parameter.detach().clone()
    for name, parameter in baseline_model.named_parameters()
}

# ------------------------------------------------------------
# Record Generator parameters BEFORE update
# ------------------------------------------------------------

generator_before = {
    name: parameter.detach().clone()
    for name, parameter in generator.named_parameters()
}

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

generated_target = generator(source_batch)

semantic_logits = baseline_model(
    source_batch,
    generated_target
)

semantic_loss = semantic_criterion(
    semantic_logits,
    label_batch
)

# ------------------------------------------------------------
# Backpropagation
# ------------------------------------------------------------

generator_optimizer.zero_grad()

semantic_loss.backward()

# ------------------------------------------------------------
# Calculate Generator gradient norm
# ------------------------------------------------------------

gradient_squared_sum = 0.0

for parameter in generator.parameters():

    if parameter.grad is not None:

        gradient_squared_sum += (
            parameter.grad.detach()
            .norm(2)
            .item() ** 2
        )

generator_gradient_norm = (
    gradient_squared_sum ** 0.5
)

# ------------------------------------------------------------
# Check gradient validity
# ------------------------------------------------------------

generator_gradients_finite = all(
    parameter.grad is None
    or torch.isfinite(parameter.grad).all()
    for parameter in generator.parameters()
)

# ------------------------------------------------------------
# Generator optimization step
# ------------------------------------------------------------

generator_optimizer.step()

# ------------------------------------------------------------
# Check whether Generator changed
# ------------------------------------------------------------

generator_changed = any(
    not torch.equal(
        generator_before[name],
        parameter.detach()
    )
    for name, parameter in generator.named_parameters()
)

# ------------------------------------------------------------
# Check whether baseline changed
# ------------------------------------------------------------

baseline_changed = any(
    not torch.equal(
        baseline_before[name],
        parameter.detach()
    )
    for name, parameter in baseline_model.named_parameters()
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("========== SINGLE-BATCH GRADIENT CHECK ==========\n")

print("Source batch shape       :", tuple(source_batch.shape))
print("Real target shape       :", tuple(target_batch.shape))
print("Generated target shape  :", tuple(generated_target.shape))
print("Labels shape            :", tuple(label_batch.shape))

print("\nSemantic loss            :",
      float(semantic_loss))

print("Generator gradient norm :",
      generator_gradient_norm)

print("\nGenerator gradients finite:",
      generator_gradients_finite)

print("Generator parameters changed:",
      generator_changed)

print("Baseline parameters changed :",
      baseline_changed)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert generated_target.shape == (32, 32)
assert semantic_logits.shape == (32,)
assert label_batch.shape == (32,)

assert torch.isfinite(semantic_loss)
assert np.isfinite(generator_gradient_norm)

assert generator_gradient_norm > 0.0
assert generator_gradients_finite

assert generator_changed
assert not baseline_changed

print("\nGenerated output shape: PASSED")
print("Semantic gradient flow: PASSED")
print("Gradient finite check: PASSED")
print("Generator update: PASSED")
print("Baseline frozen check: PASSED")

print("\nSingle-batch semantic training path: PASSED")

========== SINGLE-BATCH GRADIENT CHECK ==========

Source batch shape       : (32, 32)
Real target shape       : (32, 32)
Generated target shape  : (32, 32)
Labels shape            : (32,)

Semantic loss            : 0.7298988103866577
Generator gradient norm : 0.9843386339998542

Generator gradients finite: True
Generator parameters changed: True
Baseline parameters changed : False

Generated output shape: PASSED
Semantic gradient flow: PASSED
Gradient finite check: PASSED
Generator update: PASSED
Baseline frozen check: PASSED

Single-batch semantic training path: PASSED


C:\Users\shasw\AppData\Local\Temp\ipykernel_1800\3562477437.py:153: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  float(semantic_loss))


In [13]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 13: Clean Reset Before Official Training
# ============================================================

# ------------------------------------------------------------
# Reset all random seeds
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Fresh Generator
# ------------------------------------------------------------

generator = Generator(
    input_dim=32,
    output_dim=32
).to(device)

generator.train()

# ------------------------------------------------------------
# Fresh optimizer
# ------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# Reset deterministic DataLoader shuffle generator
# ------------------------------------------------------------

train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# ------------------------------------------------------------
# Ensure baseline remains frozen
# ------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# ------------------------------------------------------------
# Parameter counts
# ------------------------------------------------------------

generator_total_parameters = sum(
    p.numel()
    for p in generator.parameters()
)

generator_trainable_parameters = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

baseline_trainable_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
    if p.requires_grad
)

# ------------------------------------------------------------
# Display reset status
# ------------------------------------------------------------

print("========== CLEAN TRAINING RESET ==========\n")

print("Seed:", SEED)

print("\nGenerator:")
print("  Total parameters    :", generator_total_parameters)
print("  Trainable parameters:", generator_trainable_parameters)
print("  Mode                :", "train" if generator.training else "eval")

print("\nBaseline:")
print("  Trainable parameters:",
      baseline_trainable_parameters)
print("  Mode                :",
      "train" if baseline_model.training else "eval")

print("\nOptimizer:")
print("  Type       : Adam")
print("  Learning rate:", LEARNING_RATE)

print("\nDataLoader:")
print("  Training batches  :", len(train_loader))
print("  Validation batches:", len(validation_loader))
print("  Batch size        :", BATCH_SIZE)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert generator_total_parameters == 20768
assert generator_trainable_parameters == 20768

assert baseline_trainable_parameters == 0
assert not baseline_model.training
assert generator.training

assert len(train_loader) == 56
assert len(validation_loader) == 14

assert (
    generator_optimizer.param_groups[0]["lr"]
    == LEARNING_RATE
)

print("\nGenerator reset: PASSED")
print("Optimizer reset: PASSED")
print("DataLoader reset: PASSED")
print("Baseline frozen: PASSED")

print("\nClean pre-training state: PASSED")

========== CLEAN TRAINING RESET ==========

Seed: 42

Generator:
  Total parameters    : 20768
  Trainable parameters: 20768
  Mode                : train

Baseline:
  Trainable parameters: 0
  Mode                : eval

Optimizer:
  Type       : Adam
  Learning rate: 0.001

DataLoader:
  Training batches  : 56
  Validation batches: 14
  Batch size        : 32

Generator reset: PASSED
Optimizer reset: PASSED
DataLoader reset: PASSED
Baseline frozen: PASSED

Clean pre-training state: PASSED


In [14]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 14: Official Experiment Configuration
# ============================================================

NUM_EPOCHS = 50

MODEL_SELECTION_METRIC = "validation_semantic_roc_auc"

experiment_config = {
    "phase": "03.06",
    "experiment": "Experiment 3 - Semantic-Only Generator Training",
    "experiment_name": "experiment_03_semantic_only",

    "purpose": (
        "Diagnostic ablation to determine whether the Generator "
        "can learn the frozen baseline semantic relationship "
        "without adversarial training."
    ),

    "hypothesis": (
        "Removing the adversarial objective may allow the Generator "
        "to learn the semantic relationship more effectively."
    ),

    "dataset": {
        "source": "Phase 02.08 representation-aware split",
        "train_pairs": 1776,
        "validation_pairs": 448,
        "embedding_dimension": 32,
        "train_class_0": 888,
        "train_class_1": 888,
        "validation_class_0": 224,
        "validation_class_1": 224
    },

    "generator": {
        "architecture": "32 -> 64 -> 128 -> 64 -> 32",
        "total_parameters": 20768,
        "trainable_parameters": 20768
    },

    "baseline": {
        "architecture": "128 -> 128 -> 64 -> 1",
        "total_parameters": 24833,
        "trainable_parameters": 0,
        "frozen": True,
        "reference_validation_roc_auc": 0.6626674107142857
    },

    "training": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "seed": SEED
    },

    "loss": {
        "semantic_loss": "BCEWithLogitsLoss",
        "semantic_weight": 1.0,
        "adversarial_loss": "disabled",
        "adversarial_weight": 0.0,
        "generator_objective": "L_G = 1.0 * L_semantic"
    },

    "model_selection": {
        "metric": MODEL_SELECTION_METRIC,
        "direction": "maximize"
    },

    "controlled_change_from_experiment_02": {
        "experiment_02": "L_G = 0.1 * L_adv + 1.0 * L_semantic",
        "experiment_03": "L_G = 1.0 * L_semantic",
        "changed_component": "Adversarial loss removed",
        "unchanged_components": [
            "dataset",
            "generator architecture",
            "baseline architecture",
            "learning rate",
            "batch size",
            "seed",
            "number of epochs"
        ]
    }
}

# ------------------------------------------------------------
# Save configuration
# ------------------------------------------------------------

config_path = (
    experiment_03_root
    / "03_06_gan_experiment_03_configuration.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=4
    )

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("========== EXPERIMENT 3 CONFIGURATION ==========\n")

print("Experiment:")
print("  Experiment 3 - Semantic-Only Generator Training")

print("\nTraining:")
print("  Epochs       :", NUM_EPOCHS)
print("  Batch size   :", BATCH_SIZE)
print("  Learning rate:", LEARNING_RATE)
print("  Optimizer    : Adam")
print("  Seed         :", SEED)

print("\nGenerator:")
print("  Architecture : 32 -> 64 -> 128 -> 64 -> 32")
print("  Parameters   :", 20768)

print("\nLoss:")
print("  Semantic     : BCEWithLogitsLoss")
print("  Weight       : 1.0")
print("  Adversarial  : DISABLED")

print("\nSelection metric:")
print(" ", MODEL_SELECTION_METRIC)

print("\nConfiguration saved to:")
print(config_path)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert NUM_EPOCHS == 50
assert BATCH_SIZE == 32
assert LEARNING_RATE == 0.001
assert SEED == 42

assert experiment_config["loss"]["adversarial_weight"] == 0.0
assert experiment_config["loss"]["semantic_weight"] == 1.0

assert config_path.exists()

print("\nConfiguration verification: PASSED")
print("Experiment 3 is ready for official training.")

========== EXPERIMENT 3 CONFIGURATION ==========

Experiment:
  Experiment 3 - Semantic-Only Generator Training

Training:
  Epochs       : 50
  Batch size   : 32
  Learning rate: 0.001
  Optimizer    : Adam
  Seed         : 42

Generator:
  Architecture : 32 -> 64 -> 128 -> 64 -> 32
  Parameters   : 20768

Loss:
  Semantic     : BCEWithLogitsLoss
  Weight       : 1.0
  Adversarial  : DISABLED

Selection metric:
  validation_semantic_roc_auc

Configuration saved to:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\experiment_03_semantic_only\03_06_gan_experiment_03_configuration.json

Configuration verification: PASSED
Experiment 3 is ready for official training.


In [15]:
# ============================================================
# EXPERIMENT 3 — SEMANTIC-ONLY GENERATOR TRAINING
# Cell 15: Official 50-Epoch Training
# ============================================================

import time

# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

NUM_EPOCHS = 50

best_val_roc_auc = -np.inf
best_epoch = 0

training_history = {
    "epoch": [],
    "train_semantic_loss": [],
    "validation_semantic_loss": [],
    "validation_roc_auc": [],
    "validation_accuracy": [],
    "validation_balanced_accuracy": [],
    "validation_precision": [],
    "validation_recall": [],
    "validation_f1": []
}

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

best_generator_path = (
    checkpoints_dir
    / "03_06_gan_experiment_03_best_generator.pt"
)

final_generator_path = (
    checkpoints_dir
    / "03_06_gan_experiment_03_final_generator.pt"
)

history_path = (
    checkpoints_dir
    / "03_06_gan_experiment_03_training_history.pt"
)

print("========== EXPERIMENT 3 TRAINING ==========\n")

print("Objective:")
print("  L_G = 1.0 × L_semantic")

print("\nEpochs       :", NUM_EPOCHS)
print("Batch size   :", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Seed         :", SEED)

print("\nStarting training...\n")

training_start_time = time.time()

# ============================================================
# EPOCH LOOP
# ============================================================

for epoch in range(1, NUM_EPOCHS + 1):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    generator.train()

    train_loss_sum = 0.0
    train_sample_count = 0

    for source_batch, _, label_batch in train_loader:

        source_batch = source_batch.to(device)
        label_batch = label_batch.to(device)

        # Generate target-like representation
        generated_target = generator(
            source_batch
        )

        # Frozen baseline evaluates semantic relationship
        semantic_logits = baseline_model(
            source_batch,
            generated_target
        )

        # Semantic-only loss
        semantic_loss = semantic_criterion(
            semantic_logits,
            label_batch
        )

        # Generator update
        generator_optimizer.zero_grad()

        semantic_loss.backward()

        generator_optimizer.step()

        # Accumulate sample-weighted loss
        batch_size_actual = source_batch.size(0)

        train_loss_sum += (
            semantic_loss.detach().item()
            * batch_size_actual
        )

        train_sample_count += batch_size_actual

    epoch_train_semantic_loss = (
        train_loss_sum / train_sample_count
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    generator.eval()

    validation_loss_sum = 0.0
    validation_sample_count = 0

    validation_labels = []
    validation_probabilities = []

    with torch.no_grad():

        for source_batch, _, label_batch in validation_loader:

            source_batch = source_batch.to(device)
            label_batch = label_batch.to(device)

            # Generate target-like representation
            generated_target = generator(
                source_batch
            )

            # Frozen baseline semantic evaluation
            semantic_logits = baseline_model(
                source_batch,
                generated_target
            )

            semantic_loss = semantic_criterion(
                semantic_logits,
                label_batch
            )

            probabilities = torch.sigmoid(
                semantic_logits
            )

            batch_size_actual = source_batch.size(0)

            validation_loss_sum += (
                semantic_loss.item()
                * batch_size_actual
            )

            validation_sample_count += batch_size_actual

            validation_labels.extend(
                label_batch.cpu().numpy().tolist()
            )

            validation_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

    epoch_validation_semantic_loss = (
        validation_loss_sum
        / validation_sample_count
    )

    validation_labels_np = np.asarray(
        validation_labels
    )

    validation_probabilities_np = np.asarray(
        validation_probabilities
    )

    # --------------------------------------------------------
    # Validation ROC-AUC
    # --------------------------------------------------------

    epoch_val_roc_auc = roc_auc_score(
        validation_labels_np,
        validation_probabilities_np
    )

    # --------------------------------------------------------
    # Threshold-based metrics
    # --------------------------------------------------------

    validation_predictions = (
        validation_probabilities_np >= 0.5
    ).astype(int)

    epoch_val_accuracy = accuracy_score(
        validation_labels_np,
        validation_predictions
    )

    epoch_val_balanced_accuracy = (
        balanced_accuracy_score(
            validation_labels_np,
            validation_predictions
        )
    )

    epoch_val_precision = precision_score(
        validation_labels_np,
        validation_predictions,
        zero_division=0
    )

    epoch_val_recall = recall_score(
        validation_labels_np,
        validation_predictions,
        zero_division=0
    )

    epoch_val_f1 = f1_score(
        validation_labels_np,
        validation_predictions,
        zero_division=0
    )

    # --------------------------------------------------------
    # Record history
    # --------------------------------------------------------

    training_history["epoch"].append(epoch)
    training_history["train_semantic_loss"].append(
        epoch_train_semantic_loss
    )
    training_history["validation_semantic_loss"].append(
        epoch_validation_semantic_loss
    )
    training_history["validation_roc_auc"].append(
        epoch_val_roc_auc
    )
    training_history["validation_accuracy"].append(
        epoch_val_accuracy
    )
    training_history["validation_balanced_accuracy"].append(
        epoch_val_balanced_accuracy
    )
    training_history["validation_precision"].append(
        epoch_val_precision
    )
    training_history["validation_recall"].append(
        epoch_val_recall
    )
    training_history["validation_f1"].append(
        epoch_val_f1
    )

    # --------------------------------------------------------
    # Best model selection
    # --------------------------------------------------------

    is_best = (
        epoch_val_roc_auc > best_val_roc_auc
    )

    if is_best:

        best_val_roc_auc = epoch_val_roc_auc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": generator.state_dict(),
                "best_val_roc_auc": best_val_roc_auc,
                "seed": SEED,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "num_epochs": NUM_EPOCHS,
                "semantic_loss_weight": SEMANTIC_LOSS_WEIGHT,
                "adversarial_loss_weight": 0.0,
                "baseline_checkpoint": str(
                    baseline_checkpoint_path
                )
            },
            best_generator_path
        )

    # --------------------------------------------------------
    # Console output
    # --------------------------------------------------------

    best_marker = "  <-- BEST" if is_best else ""

    print(
        f"Epoch {epoch:02d} | "
        f"Train Sem {epoch_train_semantic_loss:.4f} | "
        f"Val Loss {epoch_validation_semantic_loss:.4f} | "
        f"Val AUC {epoch_val_roc_auc:.4f} | "
        f"Acc {epoch_val_accuracy:.4f} | "
        f"F1 {epoch_val_f1:.4f}"
        f"{best_marker}"
    )

# ============================================================
# SAVE FINAL GENERATOR
# ============================================================

training_elapsed_time = (
    time.time() - training_start_time
)

torch.save(
    {
        "epoch": NUM_EPOCHS,
        "model_state_dict": generator.state_dict(),
        "best_val_roc_auc": best_val_roc_auc,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "semantic_loss_weight": SEMANTIC_LOSS_WEIGHT,
        "adversarial_loss_weight": 0.0
    },
    final_generator_path
)

# ------------------------------------------------------------
# Save training history
# ------------------------------------------------------------

torch.save(
    training_history,
    history_path
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n========== TRAINING COMPLETE ==========\n")

print(
    f"Training time: "
    f"{training_elapsed_time:.2f} seconds"
)

print(
    f"Best validation ROC-AUC: "
    f"{best_val_roc_auc:.10f}"
)

print(
    f"Best epoch: {best_epoch}"
)

print("\nBest Generator:")
print(best_generator_path)

print("\nFinal Generator:")
print(final_generator_path)

print("\nTraining history:")
print(history_path)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(training_history["epoch"]) == NUM_EPOCHS
assert len(training_history["train_semantic_loss"]) == NUM_EPOCHS
assert len(training_history["validation_semantic_loss"]) == NUM_EPOCHS
assert len(training_history["validation_roc_auc"]) == NUM_EPOCHS

assert np.isfinite(
    training_history["train_semantic_loss"]
).all()

assert np.isfinite(
    training_history["validation_semantic_loss"]
).all()

assert np.isfinite(
    training_history["validation_roc_auc"]
).all()

assert best_generator_path.exists()
assert final_generator_path.exists()
assert history_path.exists()

print("\nTraining history integrity: PASSED")
print("Best checkpoint saved: PASSED")
print("Final checkpoint saved: PASSED")
print("\nExperiment 3 official training: COMPLETE")

========== EXPERIMENT 3 TRAINING ==========

Objective:
  L_G = 1.0 × L_semantic

Epochs       : 50
Batch size   : 32
Learning rate: 0.001
Seed         : 42

Starting training...

Epoch 01 | Train Sem 0.7099 | Val Loss 0.7004 | Val AUC 0.5000 | Acc 0.5000 | F1 0.4483  <-- BEST
Epoch 02 | Train Sem 0.6986 | Val Loss 0.6965 | Val AUC 0.5000 | Acc 0.5000 | F1 0.5447
Epoch 03 | Train Sem 0.6971 | Val Loss 0.6968 | Val AUC 0.5000 | Acc 0.5000 | F1 0.2632
Epoch 04 | Train Sem 0.6961 | Val Loss 0.7005 | Val AUC 0.5000 | Acc 0.5000 | F1 0.6647
Epoch 05 | Train Sem 0.6984 | Val Loss 0.6952 | Val AUC 0.5000 | Acc 0.5000 | F1 0.5897
Epoch 06 | Train Sem 0.6983 | Val Loss 0.6956 | Val AUC 0.5000 | Acc 0.5000 | F1 0.2168
Epoch 07 | Train Sem 0.6957 | Val Loss 0.6951 | Val AUC 0.5000 | Acc 0.5000 | F1 0.6111
Epoch 08 | Train Sem 0.6957 | Val Loss 0.6955 | Val AUC 0.5000 | Acc 0.5000 | F1 0.2533
Epoch 09 | Train Sem 0.6962 | Val Loss 0.6944 | Val AUC 0.5000 | Acc 0.5000 | F1 0.6254
Epoch 10 | Train S